---
# NOTEBOOK 01: INDEXAR DATASETS
---

Objetivo: Generar archivos CSV que cataloguen todas las imágenesde RivAIrSet y AqUavplant con sus metadatos y rutas a máscaras.

Salida:

- data/metadata/river_water_index.csv
- data/metadata/aquavplant_index.csv

----

# 1. Importaciones y configuración

In [24]:
import os
import csv
import pandas as pd
from pathlib import Path

# Crear carpeta de metadata si no existe
os.makedirs('data/metadata', exist_ok=True)
os.chdir(r'D:\proyecto_eutrofizacion')
print("Configuración inicial lista")

print(os.getcwd()) 

Configuración inicial lista
D:\proyecto_eutrofizacion


# 2. Función para indexar RivAIrSet

In [25]:
def indexar_river_water(carpeta_base, salida_csv):
    """
    Indexa todas las imágenes del dataset RivAIrSet.
    
    Estructura esperada:
    river_water_dataset/
    ├── 2019/
    │   ├── images/
    │   │   ├── DJI_11.JPG
    │   │   └── ...
    │   └── labels_segmentation/
    │       ├── DJI_11.json
    │       └── ...
    ├── 2020/
    └── ...
    
    Parámetros:
    -----------
    carpeta_base : str
        Ruta a la carpeta river_water_dataset
    salida_csv : str
        Ruta donde guardar el CSV generado
    
    Retorna:
    --------
    list : Lista de diccionarios con metadatos de cada imagen
    """
    
    filas = []
    
    # Itera sobre cada año (2019, 2020, 2021, ..., 2024)
    for anio in sorted(os.listdir(carpeta_base)):
        ruta_anio = os.path.join(carpeta_base, anio)
        
        # Solo procesa si es una carpeta
        if not os.path.isdir(ruta_anio):
            continue
        
        print(f"Procesando año {anio}...")
        
        # Rutas a las subcarpetas
        ruta_images = os.path.join(ruta_anio, 'images')
        ruta_labels = os.path.join(ruta_anio, 'labels_segmentation')
        
        # Verifica que existan
        if not os.path.isdir(ruta_images):
            print(f"  ⚠️  No encontrada carpeta images/ en {anio}")
            continue
        
        # Itera sobre cada imagen JPG
        for archivo in sorted(os.listdir(ruta_images)):
            if archivo.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')):
                
                # Nombre sin extensión (para buscar el JSON correspondiente)
                nombre_sin_ext = os.path.splitext(archivo)[0]
                
                # Ruta del JSON de segmentación de agua (si existe)
                ruta_json = os.path.join(ruta_labels, f"{nombre_sin_ext}.json")
                has_mask = os.path.exists(ruta_json)
                
                # Añade una fila al índice
                filas.append({
                    'filepath': os.path.join(ruta_images, archivo),
                    'source': 'river_water_dataset',
                    'group': anio,                    # Grupo para split por grupo
                    'has_gps': True,
                    'site': 'basento_river',
                    'segmentation_mask_path': ruta_json if has_mask else None,
                    'mask_type': 'water_polygon_labelimg',
                    'mask_format': 'polygon'
                })
        
        print(f"  ✓ {len([f for f in filas if f['group'] == anio])} imágenes en {anio}")
    
    # Escribe el CSV
    with open(salida_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'filepath', 'source', 'group', 'has_gps', 'site',
            'segmentation_mask_path', 'mask_type', 'mask_format'
        ])
        writer.writeheader()
        writer.writerows(filas)
    
    # Resumen
    print(f"\n{'='*70}")
    print(f"✓ RIVER_WATER_DATASET indexado")
    print(f"  Total imágenes: {len(filas)}")
    print(f"  Imágenes con máscara de agua (JSON): {sum(1 for f in filas if f['segmentation_mask_path'])}")
    print(f"  CSV guardado en: {salida_csv}")
    print(f"{'='*70}\n")
    
    return filas

# 3. Función para indexar AqUavplant

In [26]:
def indexar_aquavplant(carpeta_base, salida_csv):
    """
    Indexa todas las imágenes del dataset AqUavplant.
    
    Estructura esperada (DESPUÉS de reorganizar):
    aquavplant/
    ├── Shapla_Bil_1/
    │   ├── images/
    │   │   ├── Image4.jpg
    │   │   └── ...
    │   ├── masks_binary/
    │   │   ├── Image4_binaryMask.png
    │   │   └── ...
    │   └── masks_multiclass/
    │       ├── Image4_multiclassMask.png
    │       └── ...
    ├── Shapla_Bil_2/
    └── ... (otros sitios)
    
    Parámetros:
    -----------
    carpeta_base : str
        Ruta a la carpeta aquavplant
    salida_csv : str
        Ruta donde guardar el CSV generado
    
    Retorna:
    --------
    list : Lista de diccionarios con metadatos de cada imagen
    """
    
    filas = []
    
    # Itera sobre cada sitio
    for sitio in sorted(os.listdir(carpeta_base)):
        ruta_sitio = os.path.join(carpeta_base, sitio)
        
        # Solo procesa si es una carpeta
        if not os.path.isdir(ruta_sitio):
            continue
        
        print(f"Procesando sitio {sitio}...")
        
        # Rutas a las subcarpetas
        ruta_images = os.path.join(ruta_sitio, 'images')
        ruta_masks_binary = os.path.join(ruta_sitio, 'masks_binary')
        ruta_masks_multiclass = os.path.join(ruta_sitio, 'masks_multiclass')
        
        # Verifica que existan (si no, probablemente aún no fue reorganizado)
        if not all(os.path.isdir(r) for r in [ruta_images, ruta_masks_binary, ruta_masks_multiclass]):
            print(f"  ⚠️  {sitio} no tiene estructura reorganizada (falta images/, masks_binary/ y/o masks_multiclass/)")
            print(f"      Por favor ejecuta primero: reorganizar_aquavplant('data/raw/external/aquavplant')")
            continue
        
        # Itera sobre cada imagen JPG
        for archivo in sorted(os.listdir(ruta_images)):
            if archivo.lower().endswith(('.jpg', '.jpeg', '.png', '.tif')):
                
                # Nombre sin extensión
                nombre_sin_ext = os.path.splitext(archivo)[0]
                
                # Rutas de las máscaras
                ruta_mask_binary = os.path.join(ruta_masks_binary, f"{nombre_sin_ext}_binaryMask.png")
                ruta_mask_multiclass = os.path.join(ruta_masks_multiclass, f"{nombre_sin_ext}_multiclassMask.png")
                
                # Verifica si existen
                has_binary = os.path.exists(ruta_mask_binary)
                has_multiclass = os.path.exists(ruta_mask_multiclass)
                
                # Añade una fila al índice
                filas.append({
                    'filepath': os.path.join(ruta_images, archivo),
                    'source': 'aquavplant',
                    'group': sitio,                   # Grupo para split por grupo
                    'has_gps': False,
                    'site': sitio,
                    'mask_binary_path': ruta_mask_binary if has_binary else None,
                    'mask_multiclass_path': ruta_mask_multiclass if has_multiclass else None,
                    'mask_type': 'aquatic_plants_multiclass',
                    'mask_format': 'png_indexed',
                    'num_classes': 31
                })
        
        imágenes_sitio = len([f for f in filas if f['group'] == sitio])
        imágenes_con_multiclass = len([f for f in filas if f['group'] == sitio and f['mask_multiclass_path']])
        print(f"  ✓ {imágenes_sitio} imágenes ({imágenes_con_multiclass} con máscara multiclase)")
    
    # Escribe el CSV
    with open(salida_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'filepath', 'source', 'group', 'has_gps', 'site',
            'mask_binary_path', 'mask_multiclass_path', 'mask_type', 'mask_format', 'num_classes'
        ])
        writer.writeheader()
        writer.writerows(filas)
    
    # Resumen
    print(f"\n{'='*70}")
    print(f"✓ AQUAVPLANT indexado")
    print(f"  Total imágenes: {len(filas)}")
    print(f"  Imágenes con máscara binaria: {sum(1 for f in filas if f['mask_binary_path'])}")
    print(f"  Imágenes con máscara multiclase (31 especies): {sum(1 for f in filas if f['mask_multiclass_path'])}")
    print(f"  CSV guardado en: {salida_csv}")
    print(f"{'='*70}\n")
    
    return filas

# 4. Ejecutar indexación de RivAIrSet

In [27]:
print("INICIANDO INDEXACIÓN DE DATASETS")
print("\n")

try:
    filas_river = indexar_river_water(
        'data/raw/external/river_water_dataset',
        'data/metadata/river_water_index.csv'
    )
except Exception as e:
    print(f"❌ Error al indexar RivAIrSet: {e}")
    filas_river = []

INICIANDO INDEXACIÓN DE DATASETS


Procesando año 2019...
  ✓ 2025 imágenes en 2019
Procesando año 2020...
  ✓ 1981 imágenes en 2020
Procesando año 2021...
  ✓ 1574 imágenes en 2021
Procesando año 2022...
  ✓ 697 imágenes en 2022
Procesando año 2023...
  ✓ 1046 imágenes en 2023
Procesando año 2024...
  ✓ 307 imágenes en 2024

✓ RIVER_WATER_DATASET indexado
  Total imágenes: 7630
  Imágenes con máscara de agua (JSON): 7630
  CSV guardado en: data/metadata/river_water_index.csv



# 5. Ejecutar indexación de AqUavplant

In [28]:
try:
    filas_aqua = indexar_aquavplant(
        'data/raw/external/aquavplant',
        'data/metadata/aquavplant_index.csv'
    )
except Exception as e:
    print(f"❌ Error al indexar AqUavplant: {e}")
    filas_aqua = []

Procesando sitio BAU Botanical Garden...
  ✓ 56 imágenes (0 con máscara multiclase)
Procesando sitio BAU Museum...
  ✓ 23 imágenes (0 con máscara multiclase)
Procesando sitio Shapla Bil 1...
  ✓ 35 imágenes (0 con máscara multiclase)
Procesando sitio Shapla Bil 2...
  ✓ 15 imágenes (0 con máscara multiclase)
Procesando sitio Shapla Bil 3...
  ✓ 26 imágenes (0 con máscara multiclase)
Procesando sitio Shapla Bil 4...
  ✓ 25 imágenes (0 con máscara multiclase)
Procesando sitio Zinda Park 1...
  ✓ 6 imágenes (0 con máscara multiclase)
Procesando sitio Zinda Park 2...
  ✓ 6 imágenes (0 con máscara multiclase)
Procesando sitio Zinda Park 3...
  ✓ 5 imágenes (0 con máscara multiclase)

✓ AQUAVPLANT indexado
  Total imágenes: 197
  Imágenes con máscara binaria: 0
  Imágenes con máscara multiclase (31 especies): 0
  CSV guardado en: data/metadata/aquavplant_index.csv



# 6. Resumen combinado

In [29]:
print("\n" + "="*70)
print("RESUMEN FINAL DE INDEXACIÓN")
print("="*70)

total_imágenes = len(filas_river) + len(filas_aqua)
print(f"Total de imágenes en ambos datasets: {total_imágenes}")
print(f"  - RivAIrSet: {len(filas_river)} imágenes")
print(f"  - AqUavplant: {len(filas_aqua)} imágenes")

# Cargar y mostrar primeras líneas de ambos CSVs
print(f"\n{'='*70}")
print("PREVISUALIZACIÓN DE ARCHIVOS GENERADOS")
print(f"{'='*70}\n")

if os.path.exists('data/metadata/river_water_index.csv'):
    print("River Water Index (primeras 3 filas):")
    df_river = pd.read_csv('data/metadata/river_water_index.csv')
    print(df_river[['filepath', 'group', 'mask_type']].head(3).to_string(index=False))
    print(f"Shape: {df_river.shape[0]} filas, {df_river.shape[1]} columnas\n")
else:
    print("❌ river_water_index.csv no encontrado\n")

if os.path.exists('data/metadata/aquavplant_index.csv'):
    print("AqUavplant Index (primeras 3 filas):")
    df_aqua = pd.read_csv('data/metadata/aquavplant_index.csv')
    print(df_aqua[['filepath', 'group', 'mask_type']].head(3).to_string(index=False))
    print(f"Shape: {df_aqua.shape[0]} filas, {df_aqua.shape[1]} columnas\n")
else:
    print("❌ aquavplant_index.csv no encontrado\n")

print("="*70)
print("✓ INDEXACIÓN COMPLETADA")
print("="*70)


RESUMEN FINAL DE INDEXACIÓN
Total de imágenes en ambos datasets: 7827
  - RivAIrSet: 7630 imágenes
  - AqUavplant: 197 imágenes

PREVISUALIZACIÓN DE ARCHIVOS GENERADOS

River Water Index (primeras 3 filas):
                                                      filepath  group              mask_type
data/raw/external/river_water_dataset\2019\images\DJI_1000.JPG   2019 water_polygon_labelimg
data/raw/external/river_water_dataset\2019\images\DJI_1001.JPG   2019 water_polygon_labelimg
data/raw/external/river_water_dataset\2019\images\DJI_1002.JPG   2019 water_polygon_labelimg
Shape: 7630 filas, 8 columnas

AqUavplant Index (primeras 3 filas):
                                                            filepath                group                 mask_type
 data/raw/external/aquavplant\BAU Botanical Garden\images\Image1.jpg BAU Botanical Garden aquatic_plants_multiclass
data/raw/external/aquavplant\BAU Botanical Garden\images\Image10.jpg BAU Botanical Garden aquatic_plants_multiclass
data